In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"]="false"
from functools import partial
import time
import pathlib
import glob
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

In [ ]:
import jax
import jax.numpy as jnp
# jax.config.update("jax_enable_x64", True)
gpus = jax.devices()
print(gpus)

jax.config.update("jax_default_device", gpus[0])

import diffrax
import equinox as eqx
import optax

from natsort import natsorted
from haiku import PRNGSequence

import exciting_environments as excenvs

from dmpe.data_management import DataPaths
from dmpe.models.models import NeuralEulerODE, NeuralEulerODEPendulum, NeuralEulerODECartpole
from dmpe.evaluation.plotting_utils import plot_sequence, plot_feature_combinations
from dmpe.evaluation.experiment_utils import get_experiment_ids, load_experiment_results, load_all_experiment_results
from dmpe.evaluation.data_evaluation import DataEvaluator, JensenShannonDivergence
from dmpe.evaluation.model_evaluation import ModelWrapper, ModelEvaluator, PredictionComparison, NodeModelWrapper, EnvWrapper
from dmpe.evaluation.utils import default_constraint_function

from dmpe.evaluation.metrics_utils import default_jsd

from dmpe.utils.env_utils.fluid_tank_utils import setup_env as setup_fluid_tank_env
from dmpe.utils.env_utils.pendulum_utils import setup_env as setup_pendulum_env
from dmpe.utils.env_utils.cart_pole_utils import setup_env as setup_cart_pole_env

In [ ]:
from dmpe.models.model_training import ModelTrainer
from dmpe.evaluation.exp_data_model_learning import train_model_on_experiment_data, ModelExpDataResult

In [ ]:
from dmpe.related_work.random_walk import random_walk_control_law

In [ ]:
import matplotlib as mpl
from matplotlib import rc
rc('font',**{'family':'serif','serif':['Helvetica']})
mpl.rcParams['text.usetex'] = True
mpl.rcParams.update({'font.size': 10 * 2.54})
mpl.rcParams['text.latex.preamble']=r"\usepackage{bm}\usepackage{amsmath}\usepackage{upgreek}"

In [ ]:
full_column_width = 18.2
half_column_width = 8.89

def plot_jsd_model_prediction_relation(
    data_path: pathlib.Path,
    model_class: eqx.Module,
    verbose: bool = False,
    expecting_sub_folders: bool = True,
    penalty_function: callable = None,
    recompute_jsd: bool = False,
    consider_actions: bool = True,
):
    means = []
    medians = []
    jsds = []
    colors = []

    color_cycle = plt.rcParams["axes.prop_cycle"]()
    color_mapping = [next(color_cycle)["color"] for _ in range(15)]

    result_paths = (
        glob.glob(str(data_path) + "/**/*.eqx") if expecting_sub_folders else glob.glob(str(data_path) + "/*.eqx")
    )

    n_results = len(result_paths)
    print("# or results:", n_results)
    print(80 * "-")

    for result_path in tqdm(result_paths, total=len(result_paths)):
        result = ModelExpDataResult.from_file(
            filename=result_path,
            model_class=model_class,
        )
        if penalty_function is not None:
            if penalty_function(result.observations, result.actions) > 1:
                continue

        colors.append(result.n_datapoints)

        means.append(jnp.mean(jnp.array(result.model_errors), axis=0)[-1])
        medians.append(jnp.median(jnp.array(result.model_errors), axis=0)[-1])

        if recompute_jsd:
            jsd_value = default_jsd(
                result.observations,
                result.actions,
                points_per_dim=20,
                bounds=(-1, 1),
                bandwidth=0.08,  # TODO: What about this?!
                target_distribution=None,
                ca=consider_actions,
            )
        else:
            jsd_value = result.data_jsd

        jsds.append(jsd_value)

        if verbose:
            print(result.data_jsd)
            fig, _ = result.visualize()
            plt.show()
            print(80 * "-")

    fig, ax = plt.subplots(1, 1, figsize=(half_column_width, 4.5))

    ax.scatter(
        jsds, medians, s=25, marker="x", c=colors
    )  # , c=next(colors)["color"], label=f"{data_length} data points")

    # ax.set_ylabel("model prediction loss")
    # ax.set_xlabel("JSD")
    ax.set_ylabel(r"$\mathcal{L}_{\mathcal{M}}$")
    ax.set_xlabel(r"$\mathcal{L}_{\mathrm{JSD}}$")
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.grid(True, which="both", alpha=0.3)

    # legend_elements = [Patch(facecolor=color_mapping[idx], label=(idx + 1) * 1000) for idx in range(len(color_mapping))]
    # ax.legend(handles=legend_elements, title=r"\# of datapoints")

    return fig, ax

In [ ]:
# for sys_name in ["fluid_tank", "pendulum", "cart_pole"]:
#     for consider_actions in [True, False]:
#         fig, axs = plot_jsd_model_prediction_relation(
#             data_path=DataPaths().model_learning_cs_out / sys_name / "2step",
#             model_class=NeuralEulerODE,
#             verbose=False,
#             expecting_sub_folders=False,
#             recompute_jsd=True,
#             consider_actions=consider_actions
#         )
#         plt.savefig(f"JSD_LM_{sys_name}_ca_{consider_actions}.pdf", bbox_inches='tight');

## Combined plot

In [ ]:
def plot_jsd_model_prediction_relation_for_multiple_systems(
    data_paths: list[pathlib.Path],
    model_classes: list[eqx.Module],
    consider_actions: bool = True,
):
    fig, axs = plt.subplots(1,3, figsize=(full_column_width, 5))

    for sys_idx, (data_path, model_class) in enumerate(zip(data_paths, model_classes)):
        model_errors = []
        jsds = []
        colors = []

        result_paths = glob.glob(str(data_path) + "/*.eqx")

        n_results = len(result_paths)
        for result_path in tqdm(result_paths, total=len(result_paths)):
            result = ModelExpDataResult.from_file(
                filename=result_path,
                model_class=model_class,
            )
            colors.append(result.n_datapoints)
            model_errors.append(jnp.median(jnp.array(result.model_errors), axis=0)[-1])

            jsd_value = default_jsd(
                result.observations,
                result.actions,
                points_per_dim=20,
                bounds=(-1, 1),
                bandwidth=0.08,
                target_distribution=None,
                ca=consider_actions,
            )
            jsds.append(jsd_value)

        sc = axs[sys_idx].scatter(
            jsds, model_errors, s=25, marker="x", c=colors
        )
        # plt.colorbar(sc)

    axs[0].set_ylabel(r"$\mathcal{L}_{\mathcal{M}}$")
    for ax, col in zip(
        axs, 
        ["$\mathrm{fluid}$ $\mathrm{tank}$", "$\mathrm{pendulum}$", "$\mathrm{cart}$ $\mathrm{pole}$"]
    ):
        ax.set_title(col)

    for ax in axs:
        ax.set_xlabel(r"$\mathcal{L}_{\mathrm{JSD}}$")
        ax.set_xscale('log')
        ax.set_yscale('log')
        ax.grid(True, which="both", alpha=0.3)
    plt.tight_layout()
    return fig, ax

In [ ]:
for consider_actions in [True, False]:
    fig, axs = plot_jsd_model_prediction_relation_for_multiple_systems(
        data_paths=[
            DataPaths().model_learning_cs_out / sys_name / "2step"
            for sys_name in ["fluid_tank", "pendulum", "cart_pole"]
        ],
        model_classes=[
            NeuralEulerODE,
            NeuralEulerODEPendulum,
            NeuralEulerODECartpole,
        ],
        consider_actions=consider_actions,
    )
    plt.savefig(f"JSD_LM_all_systems_ca_{consider_actions}.pdf", bbox_inches='tight');